In [21]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

feature_files = sorted(PROCESSED_DIR.glob("*_features.parquet"))

print(f"Processed feature files: {len(feature_files)}")

for file in feature_files:
    print(file.name)

Processed feature files: 12
yellow_tripdata_2025-08_features.parquet
yellow_tripdata_2025-09_features.parquet
yellow_tripdata_2025-10_features.parquet
yellow_tripdata_2025-11_features.parquet
yellow_tripdata_2025-12_features.parquet
yellow_tripdata_2026-01_features.parquet
yellow_tripdata_2026-02_features.parquet
yellow_tripdata_2026-03_features.parquet
yellow_tripdata_2026-04_features.parquet
yellow_tripdata_2026-05_features.parquet
yellow_tripdata_2026-06_features.parquet
yellow_tripdata_2026-07_features.parquet


# The City in Motion — Data Profiling

## Objective

This notebook profiles the cleaned NYC TLC Yellow Taxi trip dataset used in
**The City in Motion** case study.

### Dataset Scope
- Source: NYC Taxi & Limousine Commission (TLC)
- Taxi type: Yellow Taxi
- Period: August 2025 – July 2026
- Analytical records: 45,676,880 trips

### Profiling Objectives
1. Understand the structure of the analytical dataset
2. Validate data types and key fields
3. Examine missing values
4. Review numerical distributions
5. Validate the engineered features
6. Identify important data-quality observations

In [22]:
import pyarrow.dataset as ds

dataset = ds.dataset(
    feature_files,
    format="parquet"
)

print("Dataset schema:")
print(dataset.schema)

Dataset schema:
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
tip_amount: double
tolls_amount: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double
pickup_date: date32[day]
pickup_hour: int64
pickup_day: int64
pickup_month: int64
pickup_year: int64
day_of_week: string
is_weekend: bool
trip_duration_minutes: int64
fare_per_mile: double
tip_percentage: double


In [23]:
total_rows = dataset.count_rows()

print(f"Total analytical trips: {total_rows:,}")


Total analytical trips: 45,676,880


In [24]:
import pyarrow.compute as pc

null_counts = {}

for field in dataset.schema:
    column = field.name
    table = dataset.to_table(columns=[column])
    null_counts[column] = table[column].null_count

null_counts

{'VendorID': 0,
 'tpep_pickup_datetime': 0,
 'tpep_dropoff_datetime': 0,
 'passenger_count': 10990659,
 'trip_distance': 0,
 'PULocationID': 0,
 'DOLocationID': 0,
 'payment_type': 0,
 'fare_amount': 0,
 'tip_amount': 0,
 'tolls_amount': 0,
 'total_amount': 0,
 'congestion_surcharge': 10990659,
 'Airport_fee': 10990659,
 'cbd_congestion_fee': 0,
 'pickup_date': 0,
 'pickup_hour': 0,
 'pickup_day': 0,
 'pickup_month': 0,
 'pickup_year': 0,
 'day_of_week': 0,
 'is_weekend': 0,
 'trip_duration_minutes': 0,
 'fare_per_mile': 1330584,
 'tip_percentage': 23021}

In [ ]:
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
    "trip_duration_minutes",
    "fare_per_mile",
    "tip_percentage"
]

summary = dataset.to_table(
    columns=numeric_columns
).to_pandas().describe().T

summary

In [ ]:
checks = {
    "trip_distance < 0": pc.sum(pc.cast(pc.less(dataset.to_table(columns=["trip_distance"])["trip_distance"], 0), pa.int64())),
    "fare_amount < 0": pc.sum(pc.cast(pc.less(dataset.to_table(columns=["fare_amount"])["fare_amount"], 0), pa.int64())),
    "total_amount < 0": pc.sum(pc.cast(pc.less(dataset.to_table(columns=["total_amount"])["total_amount"], 0), pa.int64())),
    "trip_duration < 0": pc.sum(pc.cast(pc.less(dataset.to_table(columns=["trip_duration_minutes"])["trip_duration_minutes"], 0), pa.int64())),
    "trip_duration > 1440": pc.sum(pc.cast(pc.greater(dataset.to_table(columns=["trip_duration_minutes"])["trip_duration_minutes"], 1440), pa.int64())),
}

checks

{'trip_distance < 0': <pyarrow.Int64Scalar: 0>,
 'fare_amount < 0': <pyarrow.Int64Scalar: 0>,
 'total_amount < 0': <pyarrow.Int64Scalar: 0>,
 'trip_duration < 0': <pyarrow.Int64Scalar: 0>,
 'trip_duration > 1440': <pyarrow.Int64Scalar: 0>}

In [ ]:
checks = {
    "trip_distance < 0": pc.sum(
        pc.cast(
            pc.less(dataset.to_table(columns=["trip_distance"])["trip_distance"], 0),
            pa.int64()
        )
    ).as_py(),

    "fare_amount < 0": pc.sum(
        pc.cast(
            pc.less(dataset.to_table(columns=["fare_amount"])["fare_amount"], 0),
            pa.int64()
        )
    ).as_py(),

    "total_amount < 0": pc.sum(
        pc.cast(
            pc.less(dataset.to_table(columns=["total_amount"])["total_amount"], 0),
            pa.int64()
        )
    ).as_py(),

    "trip_duration < 0": pc.sum(
        pc.cast(
            pc.less(dataset.to_table(columns=["trip_duration_minutes"])["trip_duration_minutes"], 0),
            pa.int64()
        )
    ).as_py(),

    "trip_duration > 1440": pc.sum(
        pc.cast(
            pc.greater(dataset.to_table(columns=["trip_duration_minutes"])["trip_duration_minutes"], 1440),
            pa.int64()
        )
    ).as_py()
}

checks

{'trip_distance < 0': 0,
 'fare_amount < 0': 0,
 'total_amount < 0': 0,
 'trip_duration < 0': 0,
 'trip_duration > 1440': 0}